This project uses Keras to create a regression model that analyses a one-deck game of BlackJack with 5 players

In [163]:
import random
from enum import Enum

class Suit(Enum):
    SPADES = "Spades"
    HEARTS = "Hearts"
    DIAMONDS = "Diamonds"
    CLUBS = "Clubs"


class CardValue(Enum):
    TWO = 2
    THREE = 3
    FOUR = 4
    FIVE = 5
    SIX = 6
    SEVEN = 7
    EIGHT = 8
    NINE = 9
    TEN = 10
    JACK = 11
    QUEEN = 12
    KING = 13
    ACE = 14


class Moves(Enum):
    HIT = 'Hit'
    STAND = 'Stand'
    DOUBLE = "Double"
    SPLIT = "Split"

class Card:
    def __init__(self, suit: Suit, value: CardValue):
        if not isinstance(suit, Suit):
            raise ValueError("suit must be of type Suit")
        if not isinstance(value, CardValue):
            raise ValueError("value must be of type CardValue")

        self.suit = suit
        self.value = value


card_suits = [
    Suit.SPADES,
    Suit.HEARTS,
    Suit.DIAMONDS,
    Suit.CLUBS
]
card_values = [
    CardValue.TWO,
    CardValue.THREE,
    CardValue.FOUR,
    CardValue.FIVE,
    CardValue.SIX,
    CardValue.SEVEN,
    CardValue.EIGHT,
    CardValue.NINE,
    CardValue.TEN,
    CardValue.JACK,
    CardValue.QUEEN,
    CardValue.KING,
    CardValue.ACE
]

class Hand:
    def __init__(self, bet: int):
        self.cards = []
        self.bet = bet
        self.able_to_hit = True
        self.is_soft = True
        self.is_blackjack = False

    def evalHand(self):
        output = 0
        number_of_aces = 0
        if [i.value for i in self.cards].count(CardValue.ACE) and sum([[i.value for i in self.cards].count(j) for j in
                                                                       [CardValue.TEN, CardValue.JACK, CardValue.QUEEN,CardValue.KING]]):
            self.is_blackjack = True
            return 21
        for card in self.cards:
            if card.value == CardValue.ACE:
                number_of_aces += 1
            else:
                output += [2,3,4,5,6,7,8,9,10,10,10,10][card_values.index(card.value)]
        for i in range(number_of_aces):
            if output + 11 > 21:
                output += 1
            else:
                output += 11
                self.is_soft = False
        return output


    def canSplit(self):
        if len(self.cards) == 2 and self.cards[0].value == self.cards[1].value:
            return True
        else:
            return False



class Player:
    def __init__(self, number: int):
        self.hands = []
        self.number = number


In [167]:
class Game:
    def __init__(self, num_players: int):
        standard_deck = []
        for i in enumerate(card_suits):
            for j in enumerate(card_values):
                standard_deck.append(Card(i[1], j[1]))
        print("Game started, deck reshuffled")
        random.shuffle(standard_deck)
        self.deck = standard_deck
        self.players = [Player(i + 1, 10) for i in range(num_players)]
        self.dealer = Player(0, 0)



    def giveCard(self, player: Player, number_of_hand: int):
        player.hands[number_of_hand-1].append(self.deck.pop(0))


    def voiceHand(self, player: Player, number_of_hand: int):
        output = "Player " + str(player.number) + ", hand " + str(number_of_hand) + ": "
        for card in player.hands[number_of_hand-1]:
            output += '23456789TJQKA'[card_values.index(card.value)]
            output += '♠♥♦♣'[card_suits.index(card.suit)] + " "
        output += "| " + str(self.evalHand(player, number_of_hand))
        if self.evalHand(player, number_of_hand) > 21:
            output += " BUST"
        print(output)


    def startGame(self):
        for _ in range(2):
            for player in self.players:
                self.giveCard(player, 1)
            self.giveCard(self.dealer, 1)
        print("Dealer's card:    " + str('23456789TJQKA'[card_values.index(self.dealer.hands[0][0].value)]) +
              '♠♥♦♣'[card_suits.index(self.dealer.hands[0][0].suit)])
        for player in self.players:
            self.voiceHand(player, 1)


In [172]:


def Hit(game, player_number: int, number_of_hand: int):
    player = game.players[player_number-1]
    if player.able_to_hit[number_of_hand-1] == True:
        game.giveCard(player, number_of_hand)
        game.voiceHand(player, number_of_hand)
        if game.evalHand(player, number_of_hand) > 21:
            player.able_to_hit[number_of_hand-1] = False
    else:
        print("Player " + str(player_number) + " cannot hit anymore at hand " + str(number_of_hand))


def Stand(game, player_number: int, number_of_hand: int):
    player = game.players[player_number-1]
    player.able_to_hit[number_of_hand-1] = False
    print("Player " + str(player_number) + " chose 'stand' at hand " + str(number_of_hand))


def Double(game, player_number: int, number_of_hand: int):
    player = game.players[player_number-1]
    player.able_to_hit[number_of_hand-1] = False
    player.bet_amount *= 2
    game.giveCard(player)
    print("Player " + str(player_number) + " chose 'double' at hand " + str(number_of_hand))
    game.voiceHand(player, number_of_hand)


def Split(game, player_number: int, number_of_hand: int):
    player = game.players[player_number-1]
    if ((player.hands[number_of_hand-1][0].value != player.hands[number_of_hand-1][1].value) or
            (len(player.hands[number_of_hand-1]) != 2) or not player.able_to_hit[number_of_hand-1]):
        print("Unable to split")
        return
    player.hands.append([player.hands[number_of_hand-1].pop(0)])
    player.bet_amount.append(game.players[player_number-1].bet_amount[number_of_hand-1])
    player.able_to_hit.append(True)

In [173]:
game = Game(5)
game.startGame()

Game started, deck reshuffled
Dealer's card:    T♣
Player 1, hand 1: 3♦ Q♥ | 13
Player 2, hand 1: 5♣ 7♥ | 12
Player 3, hand 1: 5♦ 7♠ | 12
Player 4, hand 1: 5♥ 2♦ | 7
Player 5, hand 1: A♦ 6♠ | 17


In [2]:
import numpy as np
settlements = np.zeros(shape=(5, 4))

In [6]:
settlements[1] = np.array([1,2,3,4])

In [7]:
settlements

array([[0., 0., 0., 0.],
       [1., 2., 3., 4.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.]])

In [8]:
a = Hand()

NameError: name 'Hand' is not defined